<a href="https://colab.research.google.com/github/nabilah-afrin/Computer-vision-projects/blob/main/test_time_adaptation/tent_step_by_step.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tent

In [1]:
!pip install iopath
!pip install git+https://github.com/facebookresearch/fvcore.git
!pip install git+https://github.com/fra31/auto-attack.git@a39220048b3c9f2cca9a4d3a54604793c68eca7e#egg=autoattack
!pip install git+https://github.com/RobustBench/robustbench.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31527 sha256=2d65ae7424065a55bde64f9f8756e7f093430d9ab20729c65b2a1d25ae2b477c
  Stored in directory: /root/.cache/pip/wheels/ba/5e/16/6117f8fe7e9c0c161a795e10d94645ebcf301ccbd01f66d8ec
Successfully built iopath
  Cloning https://github.com/facebookresearch/fvcore.git to /tmp/pip-req-build-tqk3l4p8
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/fvcore.git /tmp/pip-req-build-tqk3l4p8
  Resolved https://github.com/facebookresearch/fvcore.git to commit b2ac9aba9354283ec11eb8f7b8891e6f63a0902b
  Preparing metadata (setup.py) ... done
  Created wheel for fvcore: filename=fvcore-0.1.6-py3-none-any.whl size=65669 sha256=182dfc4500e1612c4978b37a19d38ab70bdab12ace91ae2c3e038f71c7220345
  Stored in directory: /tmp/pip-ephem-wheel-cache-myjmz5hw/wheels/cb/7b/f1/17

In [2]:
!git clone https://github.com/DequanWang/tent.git
%cd tent

Cloning into 'tent'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 58 (delta 28), reused 53 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 20.57 KiB | 1.87 MiB/s, done.
Resolving deltas: 100% (28/28), done.
/content/tent


In [8]:
# import logging
# import torch
# import numpy as np
import tent
# import torch.utils.data as data
# from robustbench.data import load_cifar10c
# from robustbench.model_zoo.enums import ThreatModel
# from robustbench.utils import load_model
# from robustbench.utils import clean_accuracy as accuracy

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [5]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## base model

In [6]:
base_model = load_model(
    model_name='Standard',
    dataset='cifar10',
    threat_model=ThreatModel.corruptions
).cuda()

Downloading...
From (original): https://drive.google.com/uc?id=1t98aEuzeTL8P7Kpd5DIrCoCL21BNZUhC
From (redirected): https://drive.google.com/uc?id=1t98aEuzeTL8P7Kpd5DIrCoCL21BNZUhC&confirm=t&uuid=e8795803-fdb3-42aa-9bfc-9951f991ffe7
To: /content/tent/models/cifar10/corruptions/Standard.pt
100%|██████████| 292M/292M [00:03<00:00, 74.9MB/s]


### tent confiure

In [10]:
model = tent.configure_model(base_model)
params, param_names = tent.collect_params(model)
optimizer = torch.optim.Adam(params, lr=1e-3)
tented_model = tent.Tent(model, optimizer)

In [11]:
print(param_names)

['block1.layer.0.bn1.weight', 'block1.layer.0.bn1.bias', 'block1.layer.0.bn2.weight', 'block1.layer.0.bn2.bias', 'block1.layer.1.bn1.weight', 'block1.layer.1.bn1.bias', 'block1.layer.1.bn2.weight', 'block1.layer.1.bn2.bias', 'block1.layer.2.bn1.weight', 'block1.layer.2.bn1.bias', 'block1.layer.2.bn2.weight', 'block1.layer.2.bn2.bias', 'block1.layer.3.bn1.weight', 'block1.layer.3.bn1.bias', 'block1.layer.3.bn2.weight', 'block1.layer.3.bn2.bias', 'block2.layer.0.bn1.weight', 'block2.layer.0.bn1.bias', 'block2.layer.0.bn2.weight', 'block2.layer.0.bn2.bias', 'block2.layer.1.bn1.weight', 'block2.layer.1.bn1.bias', 'block2.layer.1.bn2.weight', 'block2.layer.1.bn2.bias', 'block2.layer.2.bn1.weight', 'block2.layer.2.bn1.bias', 'block2.layer.2.bn2.weight', 'block2.layer.2.bn2.bias', 'block2.layer.3.bn1.weight', 'block2.layer.3.bn1.bias', 'block2.layer.3.bn2.weight', 'block2.layer.3.bn2.bias', 'block3.layer.0.bn1.weight', 'block3.layer.0.bn1.bias', 'block3.layer.0.bn2.weight', 'block3.layer.0.bn

In [12]:
x_test, y_test = load_cifar10c(n_examples= 10000,severity=1, data_dir='./data', shuffle=False, corruptions=['gaussian_noise'])

Starting download from https://zenodo.org/api/records/2535967/files/CIFAR-10-C.tar/content


44533it [24:58, 29.72it/s]                           


Download finished, extracting...
Downloaded and extracted.


In [15]:
x_test = x_test.cuda()
y_test = y_test.cuda()

## Adaptation

In [18]:
# tented_model.reset() #when working with multiple corruptions
corruption = 'gaussian_noise'
severity = 1
acc = accuracy(tented_model, x_test, y_test, batch_size=128)
err = 1. - acc
logger.info(f"error % [{corruption}{severity}]: {err:.2%}")

In [ ]:
torch.save(tented_model.model.state_dict(), f"tent_adapted_{corruption}_sev{severity}.pth")
logger.info(f"Saved adapted model weights")

In [19]:
torch.save(tented_model.model.state_dict(), "adapted_model.pth")

In [21]:
print(f"\n{'='*40}")
print(f"Corruption: {corruption}, Severity: {severity}")
print(f"Accuracy: {acc:.2%}")
print(f"Error: {1-acc:.2%}")
print(f"{'='*40}\n")


Corruption: gaussian_noise, Severity: 1
Accuracy: 89.24%
Error: 10.76%



In [22]:
# Method 3: Detailed batch-by-batch view
def detailed_accuracy(model, x, y, batch_size=128):
    model.eval()
    total = 0
    correct = 0

    # Calculate number of batches
    n_batches = (x.size(0) + batch_size - 1) // batch_size

    for i in range(n_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, x.size(0))

        x_batch = x[start_idx:end_idx]
        y_batch = y[start_idx:end_idx]

        outputs = model(x_batch)
        _, preds = torch.max(outputs, 1)

        batch_correct = (preds == y_batch).sum().item()
        batch_total = y_batch.size(0)

        correct += batch_correct
        total += batch_total

        # Print batch results
        print(f"Batch {i+1}/{n_batches}: "
              f"Acc = {batch_correct}/{batch_total} ({batch_correct/batch_total:.2%}) "
              f"Cumulative = {correct}/{total} ({correct/total:.2%})")

    final_acc = correct / total
    print(f"\nFINAL ACCURACY: {final_acc:.2%}")
    return final_acc

# Use detailed accuracy instead:
acc = detailed_accuracy(tented_model, x_test, y_test, batch_size=128)

Batch 1/79: Acc = 118/128 (92.19%) Cumulative = 118/128 (92.19%)
Batch 2/79: Acc = 115/128 (89.84%) Cumulative = 233/256 (91.02%)
Batch 3/79: Acc = 116/128 (90.62%) Cumulative = 349/384 (90.89%)
Batch 4/79: Acc = 114/128 (89.06%) Cumulative = 463/512 (90.43%)
Batch 5/79: Acc = 124/128 (96.88%) Cumulative = 587/640 (91.72%)
Batch 6/79: Acc = 111/128 (86.72%) Cumulative = 698/768 (90.89%)
Batch 7/79: Acc = 105/128 (82.03%) Cumulative = 803/896 (89.62%)
Batch 8/79: Acc = 113/128 (88.28%) Cumulative = 916/1024 (89.45%)
Batch 9/79: Acc = 108/128 (84.38%) Cumulative = 1024/1152 (88.89%)
Batch 10/79: Acc = 114/128 (89.06%) Cumulative = 1138/1280 (88.91%)
Batch 11/79: Acc = 114/128 (89.06%) Cumulative = 1252/1408 (88.92%)
Batch 12/79: Acc = 119/128 (92.97%) Cumulative = 1371/1536 (89.26%)
Batch 13/79: Acc = 118/128 (92.19%) Cumulative = 1489/1664 (89.48%)
Batch 14/79: Acc = 115/128 (89.84%) Cumulative = 1604/1792 (89.51%)
Batch 15/79: Acc = 105/128 (82.03%) Cumulative = 1709/1920 (89.01%)
Batc